# M1 — arXiv crawler (Colab): batch + reference theo từng batch

Mỗi batch 50 ID tạo ra **3 thứ** trong `DATA_DIR`:

| Thành phần | Nội dung |
| --- | --- |
| `batch_00000.json` | Manifest: **50 ID** của batch, số ok/error/no_tex/not_found, **và tóm tắt reference** (khóa `references`) |
| `batch_00000.tar.gz` | `papers.jsonl`, `crawl_records.jsonl`, `metrics.json` và các file `.tex`/`.bib` của mọi version |
| `reference_00000/` | Mỗi paper một file `{2603-15580}.json` chứa **mọi** reference từ Semantic Scholar (kể cả reference không có arXiv ID) |

Không có file `reference_00000.json` riêng: danh sách tổng (trạng thái, số reference, số reference có arXiv ID của từng paper) nằm ngay trong `batch_00000.json`.

Hàm `crawl_batches(n)` chạy theo thứ tự: crawl n batch → lấy reference còn thiếu → **in ra** nội dung batch và thư mục reference.

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M1-Ref"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/m1_crawling/requirements-crawler.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_arxiv_crawler.py", "tests/test_runtime_tracking.py", "tests/test_concurrent_crawler.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Chọn khoảng ID và thư mục lưu

`MAX_WORKERS = 1` (mặc định): đề yêu cầu chỉ dùng **một connection tại một thời điểm** với arXiv. Chỉ tăng khi đã xác nhận được phép. **Không đổi `BATCH_SIZE` khi đã có checkpoint.**

In [ ]:
RANGE_ID = 3           # 1..5, mỗi thành viên chọn range được giao
BATCH_SIZE = 50          # giữ nguyên khi chạy lại / Recovery
MAX_WORKERS = 1          # đề: một connection tại một thời điểm
DATA_DIR = Path(f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Range:", RANGE_ID, "| Batch size:", BATCH_SIZE, "| Workers:", MAX_WORKERS, "| Drive:", DATA_DIR)

## 2 - arXiv crawler (function definition only)

`fast_crawl(batches=n)` processes n new batches, retries temporary HTTP/network errors, and checkpoints each completed batch. It only crawls arXiv. Use `crawl_batches(n)` in section 5 to also fetch references and print the results.


In [ ]:
import time
from src.m1_crawling.runtime_tracking import (
    checkpoint_state, record_runtime, utc_now,
)


def _run_crawler(extra_args, *, mode, batches_requested):
    cmd = [
        sys.executable, "-u",
        "-m", "src.m1_crawling.run_arxiv_only",
        "--range", str(RANGE_ID),
        "--batch-size", str(BATCH_SIZE),
        "--backup-dir", str(DATA_DIR),
        "--delay", "3.1",
        "--workers", str(MAX_WORKERS),
        *extra_args
    ]

    before = checkpoint_state(DATA_DIR)
    started_at = utc_now()
    start = time.perf_counter()
    exit_code = -1

    try:
        process = subprocess.Popen(
            cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            # Hide the CLI's range/start banners; keep progress, retries, and summaries.
            if line.startswith(("Assigned range=", "Start=")):
                continue
            print(line, end="", flush=True)
        exit_code = process.wait()
    finally:
        elapsed = time.perf_counter() - start
        try:
            event = record_runtime(
                DATA_DIR, mode=mode, range_id=RANGE_ID, batch_size=BATCH_SIZE,
                batches_requested=batches_requested, elapsed_seconds=elapsed,
                exit_code=exit_code, started_at_utc=started_at, before=before,
                after=checkpoint_state(DATA_DIR),
            )
            print(f"\n[RUNTIME] {mode}: {elapsed:.1f}s ({elapsed / 60:.2f} phút), "
                  f"trạng thái={event['result']}", flush=True)
        except OSError as exc:
            print(f"[WARNING] Không lưu được runtime lên Drive: {exc}")

    # Code 2 = có source lỗi, checkpoint vẫn được lưu.
    if exit_code not in (0, 2):
        raise RuntimeError(f"Crawler failed with exit code {exit_code}")
    return exit_code


def fast_crawl(batches=1, retries=4):
    return _run_crawler(["--next", str(batches), "--retries", str(retries)],
                        mode="fast", batches_requested=batches)

## 3 — Reference (Semantic Scholar): phần còn thiếu của mỗi batch

Với mỗi `batch_XXXXX.json` hàm `crawl_references()`:

1. Đọc 50 ID từ manifest; bỏ qua ID có trạng thái `not_found` (không có paper để tra).
2. Gọi Semantic Scholar một lần cho mỗi paper (≥ 1 giây/request, có tự retry khi 429) và ghi `reference_XXXXX/{2603-15580}.json`.
3. Ghi tóm tắt vào khóa `references` của `batch_XXXXX.json` (không đụng `ids`, `archive_bytes`, `archive_sha256`).

Chạy lại an toàn: paper đã có file thì không gọi API lại. Trạng thái mỗi paper: `ok` (có reference), `empty` (S2 trả 0 reference, thường do chưa index paper mới 1–7 ngày), `error` (lỗi mạng/API, sẽ tự thử lại lần chạy sau).

**API key (khuyến nghị):** Colab → biểu tượng chìa khóa (Secrets) → thêm `S2_API_KEY` và bật quyền truy cập notebook. Không có key thì bị giới hạn 100 request / 5 phút.

In [ ]:
import json
import os
import tarfile

from src.m1_crawling import batch_references as _refs

# Optional API key from Colab Secrets (secret name: S2_API_KEY).
try:
    from google.colab import userdata
    s2_key = userdata.get("S2_API_KEY")
    if s2_key:
        os.environ["S2_API_KEY"] = s2_key
except Exception:
    # Public API quota is used when no secret is configured.
    pass

print(
    "S2 API key:",
    "có" if os.environ.get("S2_API_KEY")
    else "không (100 req / 5 phút)"
)


def reference_dir_for(batch_name):
    return _refs.reference_dir_for(DATA_DIR, batch_name)


def crawl_references(batch_names=None, retry_empty=False):
    """Lấy reference còn thiếu cho các batch đã có checkpoint hợp lệ."""
    return _refs.crawl_references(
        DATA_DIR,
        batch_names=batch_names,
        retry_empty=retry_empty,
        api_key=os.environ.get("S2_API_KEY")
    )

## 4 — In nội dung batch và thư mục reference

In [ ]:
import textwrap
from src.m1_crawling.build_tables import build_batch_tables


def show_batch(batch, max_files=8):
    """In batch_XXXXX.json, batch_XXXXX.tar.gz và thư mục reference_XXXXX."""
    name = f"batch_{batch:05d}" if isinstance(batch, int) else batch
    marker, archive = DATA_DIR / f"{name}.json", DATA_DIR / f"{name}.tar.gz"
    info = json.loads(marker.read_text(encoding="utf-8"))
    ids = info["ids"]

    print("=" * 78)
    print(f"{marker.name}: {len(ids)} ID ({ids[0]} → {ids[-1]})")
    print(textwrap.fill(", ".join(ids), 100, initial_indent="  ", subsequent_indent="  "))
    print(f"  ok={info.get('ok')}  error={info.get('failed')}  no_tex={info.get('no_tex')}  "
          f"not_found={info.get('not_found')}  runtime={info.get('runtime_seconds_this_run')}s")

    print("-" * 78)
    with tarfile.open(archive, "r:gz") as tf:
        files = [m.name for m in tf if m.isfile()]
    top = [f for f in files if not f.startswith("working/")]
    tex_bib = [f for f in files if f.lower().endswith((".tex", ".bib"))]
    print(f"{archive.name}: {archive.stat().st_size / 1024**2:.2f} MiB, {len(files)} file "
          f"({len(tex_bib)} .tex/.bib)")
    print("  file cấp cao:", ", ".join(top))

    print("-" * 78)
    ref_dir = reference_dir_for(name)
    ref_files = sorted(ref_dir.glob("*.json")) if ref_dir.is_dir() else []
    summary = info.get("references")
    print(f"{ref_dir.name}/: {len(ref_files)} file")
    if ref_files:
        print("  files:", ", ".join(path.name for path in ref_files))
        print("  all reference records:")
        for path in ref_files:
            try:
                record = json.loads(path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError) as exc:
                print(f"  - {path.name}: unreadable ({exc})")
                continue
            refs = record.get("references") or []
            print(f"  [{path.name}] status={record.get('status')} | {len(refs)} references")
            for ref in refs:
                authors = ", ".join(ref.get("ref_authors") or [])
                print(f"    - {ref.get('ref_title') or '(untitled)'} | "
                      f"authors={authors or 'unknown'} | "
                      f"date={ref.get('ref_submission_date') or 'unknown'} | "
                      f"arXiv={ref.get('ref_arxiv_id') or 'none'} | "
                      f"S2={ref.get('ref_s2_id') or 'none'}")
    cumulative = info.get("references_total")
    if cumulative:
        print("  TOTAL ALL BATCHES: "
              f"{cumulative['total_refs']} references | "
              f"{cumulative['refs_with_arxiv_id']} with arXiv ID | "
              f"{cumulative['batches_with_reference_summary']} batches")
    if summary:
        print(f"  ok={summary['papers_ok']} empty={summary['papers_empty']} "
              f"error={summary['papers_error']} skipped={summary['papers_skipped']} | "
              f"{summary['total_refs']} reference, {summary['refs_with_arxiv_id']} có arXiv ID")
        for arxiv_id, p in list(summary["per_paper"].items())[:max_files]:
            print(f"  - {arxiv_id}: {p['status']}, {p['n_refs']} ref, {p['n_with_arxiv_id']} có arXiv ID")
        if len(ids) > max_files:
            print(f"  ... ({len(ids) - max_files} paper nữa, xem khóa 'references' trong {marker.name})")
        missing = [i for i, p in summary["per_paper"].items() if p["status"] in ("empty", "error")]
        if missing:
            print(f"  CẦN XEM LẠI ({len(missing)}): {', '.join(missing)}")
    else:
        print("  Chưa lấy reference cho batch này: chạy crawl_references().")
    print("=" * 78)


def print_reference_totals():
    """Print cumulative reference totals from all batch manifests in DATA_DIR."""
    summaries = []
    for marker in sorted(DATA_DIR.glob("batch_*.json")):
        try:
            manifest = json.loads(marker.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        summary = manifest.get("references")
        if isinstance(summary, dict):
            summaries.append(summary)
    print("\n===== REFERENCES TOTAL (all batches with reference summaries) =====")
    print(f"Batches with reference summary: {len(summaries)}")
    print(f"References collected: {sum(s.get('total_refs', 0) for s in summaries)}")
    print(f"References with arXiv ID: {sum(s.get('refs_with_arxiv_id', 0) for s in summaries)}")
    print(f"Papers: ok={sum(s.get('papers_ok', 0) for s in summaries)} "
          f"empty={sum(s.get('papers_empty', 0) for s in summaries)} "
          f"error={sum(s.get('papers_error', 0) for s in summaries)} "
          f"skipped={sum(s.get('papers_skipped', 0) for s in summaries)}")

def crawl_batches(batches=1, retry_empty=False):
    """Crawl n new batches, fetch missing references, print details and totals."""
    before = {p.stem for p in DATA_DIR.glob("batch_*.json")}
    fast_crawl(batches=batches)
    newly_crawled = {p.stem for p in DATA_DIR.glob("batch_*.json")} - before
    reference_updates = set(crawl_references(retry_empty=retry_empty))
    to_show = sorted(newly_crawled | reference_updates)
    if not to_show:
        # Even when crawl/reference work is already complete, display the latest
        # existing batch so its manifest and reference folder are visible.
        existing = sorted(p.stem for p in DATA_DIR.glob("batch_*.json"))
        count = max(1, batches) if batches is not None else 1
        to_show = existing[-count:]
    if not to_show:
        print("No batch checkpoints found in DATA_DIR.")
        return []
    for name in to_show:
        show_batch(name)

    # Also create per-batch tables for existing reference folders from earlier runs.
    table_batches = set(to_show)
    existing_batches = {p.stem for p in DATA_DIR.glob("batch_*.json")}
    for ref_dir in DATA_DIR.glob("reference_*"):
        if not ref_dir.is_dir():
            continue
        batch_name = "batch_" + ref_dir.name.removeprefix("reference_")
        report_path = ref_dir / "tables" / "build_report.json"
        if batch_name in existing_batches and not report_path.is_file():
            table_batches.add(batch_name)

    for name in sorted(table_batches):
        try:
            batch_report = build_batch_tables(DATA_DIR, name)
            print(f"Batch tables: {DATA_DIR / reference_dir_for(name).name / 'tables'} | "
                  f"papers={batch_report['papers_rows']} "
                  f"references={batch_report['references_rows']}")
        except RuntimeError as exc:
            print(f"[TABLES SKIPPED] {name}: {exc}")
    print_reference_totals()
    return to_show


## 5 — Chạy

Bắt đầu với **1 batch** (50 ID đầu tiên → `batch_00000`). Sau khi kiểm tra kết quả, tăng `batches` (ví dụ 10); `batches=0` = toàn bộ batch còn thiếu. Nếu Colab ngắt, mount lại và chạy lại các cell định nghĩa hàm rồi gọi lại: batch đã xong được bỏ qua.

In [ ]:
crawl_batches(batches=1)

## 6 — Kiểm tra các checkpoint đã lưu

In [ ]:
def show_checkpoints():
    manifests = sorted(DATA_DIR.glob("batch_*.json"))
    print(f"Đã lưu {len(manifests)} checkpoint")
    for marker in manifests:
        info = json.loads(marker.read_text(encoding="utf-8"))
        archive = marker.with_suffix(".tar.gz")
        if not archive.exists():
            print("MISSING ARCHIVE:", marker.name)
            continue
        refs = info.get("references")
        ref_text = (f"ref ok={refs['papers_ok']} empty={refs['papers_empty']} "
                    f"error={refs['papers_error']}" if refs else "ref: chưa có")
        print(f"{marker.stem}: ok={info.get('ok')}, error={info.get('failed')}, "
              f"no_tex={info.get('no_tex')}, size={archive.stat().st_size/1024**2:.2f} MiB | {ref_text}")

show_checkpoints()

## 7 — Recovery: chỉ tải lại phiên bản lỗi, KHÔNG tự sao lưu bản cũ

- Mặc định `backup_before=False`: tránh sao chép toàn bộ `.tar.gz` vào `_before_recovery` và làm đầy Google Drive.
- Recovery chỉ xử lý checkpoint đã tồn tại và còn lỗi; `no_tex` không retry. `batches=1` nghĩa là tối đa **1 batch có lỗi**, không phải 1 bài báo.
- Recovery ghi lại `batch_XXXXX.json` từ đầu nên khóa `references` biến mất; hàm tự gọi `crawl_references()` sau đó để gắn lại (không gọi lại API cho paper đã có file).

In [ ]:
import shutil

def recovery_crawl(batches=10, retries=4, backup_before=False):
    before = {p.stem: p.stat().st_mtime_ns for p in DATA_DIR.glob("batch_*.json")}
    if backup_before:
        snapshot = DATA_DIR / "_before_recovery"
        snapshot.mkdir(parents=True, exist_ok=True)
        copied = 0
        for marker in sorted(DATA_DIR.glob("batch_*.json")):
            info = json.loads(marker.read_text(encoding="utf-8"))
            if int(info.get("failed", 0)) <= 0:
                continue
            archive = marker.with_suffix(".tar.gz")
            if not archive.exists():
                raise FileNotFoundError(f"Missing archive: {archive}")
            for source in (marker, archive):
                destination = snapshot / source.name
                if not destination.exists():
                    shutil.copy2(source, destination)
                    copied += 1
        print(f"Saved {copied} original checkpoint files in {snapshot}")
    code_ = _run_crawler(["--next", str(batches), "--retry-failed",
                          "--retries", str(retries)],
                         mode="recovery", batches_requested=batches)
    reference_updates = set(crawl_references())
    changed = {p.stem for p in DATA_DIR.glob("batch_*.json")
               if before.get(p.stem) != p.stat().st_mtime_ns}
    for name in sorted(changed | reference_updates):
        show_batch(name)
    return code_

# Chỉ chạy khi cần Recovery (bỏ dấu # ở dòng dưới):
# recovery_crawl(batches=100, retries=1)

## 8 — Export report: lỗi + thời gian tích lũy

- `runtime_history.csv`: từng lần Fast/Recovery và số giây thực tế.
- `summary.json`, `issues_explained_vi.csv`, `papers.csv`, `crawl_records.csv`... như bản trước. Export nhiều lần không ghi thêm runtime.

In [ ]:
from src.m1_crawling.export_reports import export_reports


def export_report(verify_sha256=False):
    summary = export_reports(DATA_DIR, verify_sha256=verify_sha256)
    print("===== REPORT =====")
    for key in ("batch_count", "total_records", "metadata_rows", "ok", "error",
                "no_tex", "not_found", "failed_version_rows"):
        print(f"{key}: {summary[key]}")
    print("Loai van de:", summary.get("issue_counts", {}))
    print("===== RUNTIME (từ khi cài bản vá) =====")
    for key in ("fast_crawl_seconds", "recovery_seconds", "total_runtime_seconds",
                "fast_crawl_runs", "recovery_attempts", "average_seconds_per_id"):
        print(f"{key}: {summary.get(key)}")
    print("Lịch sử:", DATA_DIR / "reports" / "runtime_history.csv")
    print("Checkpoint: %.2f MiB" % (summary.get("checkpoint_archive_bytes", 0) / 1024**2))
    print("Ban sao Recovery: %.2f MiB" % (summary.get("recovery_snapshot_bytes", 0) / 1024**2))
    print("Saved at:", DATA_DIR / "reports")
    for path in sorted((DATA_DIR / "reports").iterdir()):
        print(" -", path.name)
    return summary

# summary = export_report()

## 9 - Build and validate the `papers` and `references` tables

- Each batch also gets `reference_XXXXX/tables/` with `papers.json`, `references.json`, and JSONL copies.
- The final combined tables are written to `DATA_DIR/tables/`.
- The references table keeps valid arXiv IDs only; all Semantic Scholar records remain in `reference_*/`.
- Missing fields, invalid types, and duplicates are reported in `tables/validation_report.json`.


In [ ]:
from src.m1_crawling.build_tables import build_tables
from src.m1_crawling.validate_tables import validate_tables

report = build_tables(DATA_DIR)
print(f"papers: {report['papers_rows']} dòng | references (có arXiv ID): {report['references_rows']} dòng")
print("Paper chưa có dữ liệu reference:", report["references"]["papers_without_reference_data"])
validation = validate_tables(DATA_DIR / "tables")

## 10 — Sau khi chạy

- Gửi cho người xử lý dữ liệu: `batch_*.tar.gz`, `reference_*/`, `reports/papers.csv`, `reports/crawl_records.csv` và đường dẫn Drive.
- Reference nào `empty` có thể do Semantic Scholar chưa index; thử lại sau vài ngày bằng `crawl_references(retry_empty=True)`.
- Bảng `references` nộp cuối chỉ lấy reference có `ref_arxiv_id`; thư mục `reference_*/` giữ tất cả.